# Representação vetorial de textos

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/03_representacao_vetorial_de_textos.ipynb)

Os notebooks anteriores compararam quantidades entre grupos. Para comparar
documentos, precisamos primeiro decidir como um texto se torna uma estrutura
calculável.

## 1. *Bag of Words*: o que preserva e o que perde

*Bag of Words* representa um documento pelas frequências dos termos. Preserva
parte do vocabulário e da repetição, mas descarta ordem, sintaxe, ironia,
negação, voz e grande parte do contexto.

`"trabalho não é progresso"` e `"progresso não é trabalho"` recebem as mesmas
contagens sob esta representação. Isso não é erro de programação: é uma perda
deliberada do modelo.

![Documentos preservados passam por normalização e tokens, formam um vocabulário e chegam a uma matriz de documentos por termos; uma nota registra a perda de ordem e sintaxe.](imagens/03_fluxo_matriz_documento_termo.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_05'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import math
import re
import unicodedata
from collections import Counter
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd())) if str(Path.cwd()) not in sys.path else None
from IPython.display import display
from graficos import mapa_calor

dados = pd.read_csv("dados/documentos_comparacao.csv")
dados["texto"] = dados["texto_comparacao"]
stopwords = {"a", "e", "o", "de", "do", "da", "em", "nas", "também"}

def tokenizar(texto):
    normalizado = unicodedata.normalize("NFKD", texto.lower())
    sem_acentos = "".join(c for c in normalizado if not unicodedata.combining(c))
    return [t for t in re.findall(r"[a-z]+", sem_acentos) if t not in stopwords]

dados["tokens"] = dados["texto"].map(tokenizar)
dados[["id_documento", "tokens"]].head(3)

A função explicita minúsculas, remoção de acentos, tokenização e *stopwords*.
Outra regra produziria outro vocabulário. Preservamos `texto` e o identificador
para poder retornar à fonte.

## 2. Matriz documento-termo

Se $c(t,d)$ é a contagem do termo $t$ no documento $d$, a matriz é:

$$
X_{d,t}=c(t,d).
$$

Linhas representam documentos; colunas, termos; células, frequências. Zero
significa ausência segundo as regras adotadas, não irrelevância histórica.

In [ ]:
contagens = [Counter(tokens) for tokens in dados["tokens"]]
matriz_contagens = pd.DataFrame(contagens, index=dados["id_documento"]).fillna(0).astype(int)
matriz_contagens.index.name = "id_documento"
vocabulario = sorted(matriz_contagens.columns)
print("Dimensões:", matriz_contagens.shape)
print("Vocabulário:", len(vocabulario), "termos")
display(matriz_contagens.iloc[:6, :10])
mapa_calor(matriz_contagens.iloc[:8, :12], "Recorte da matriz documento-termo")

A matriz torna os documentos comparáveis, mas termos recorrentes em toda a
coleção podem dominar a contagem. TF-IDF acrescenta uma ponderação pela raridade
documental.

## 3. TF, DF, IDF e TF-IDF

Neste notebook, TF é a contagem bruta. A frequência de documento usa presença
positiva, e a convenção de IDF é suavizada:

$$
df(t)=\sum_{d=1}^{N}\mathbf{1}(X_{d,t}>0),\qquad
idf(t)=\log\left(\frac{N+1}{df(t)+1}\right)+1.
$$

Finalmente:

$$
tfidf(t,d)=X_{d,t}\times idf(t).
$$

A suavização, a base do logaritmo e a normalização variam entre implementações;
documentá-las é parte do método.

![TF mede frequência no documento, IDF mede raridade entre documentos e seu produto gera um peso; a figura alerta que peso não é importância histórica.](imagens/03_anatomia_tfidf.svg)

In [ ]:
N = len(matriz_contagens)
frequencia_documento = matriz_contagens.gt(0).sum(axis=0)
idf = ((N + 1) / (frequencia_documento + 1)).map(math.log) + 1
matriz_tfidf = matriz_contagens.mul(idf, axis=1)
documento_exemplo = "D001"
comparacao_pesos = pd.DataFrame({
    "contagem": matriz_contagens.loc[documento_exemplo],
    "df": frequencia_documento,
    "idf": idf,
    "tfidf": matriz_tfidf.loc[documento_exemplo],
}).sort_values("tfidf", ascending=False)
comparacao_pesos.head(10).round(3)

Um peso alto identifica termo frequente no documento e relativamente raro na
coleção, sob esta regra. Não prova que o termo seja tema central, conceito
histórico ou escolha consciente do autor.

## 4. Comparar períodos e coleções

Agregar vetores por período permite localizar termos distintivos, mas o período
precisa ser definido antes da inspeção. Usaremos 1890–1895 e 1896–1901; os anos
posteriores não existem neste conjunto fictício.

In [ ]:
periodo = pd.Series(
    pd.cut(dados["ano"], bins=[1889, 1895, 1901], labels=["1890–1895", "1896–1901"]).array,
    index=dados["id_documento"], name="periodo"
)
media_tfidf_periodo = matriz_tfidf.groupby(periodo, observed=True).mean()
contraste_periodos = (media_tfidf_periodo.loc["1890–1895"] - media_tfidf_periodo.loc["1896–1901"])
termos_periodo_inicial = contraste_periodos.nlargest(6).rename("favorece 1890–1895")
termos_periodo_final = contraste_periodos.nsmallest(6).rename("favorece 1896–1901")
tabela_contrastes = pd.concat([
    termos_periodo_inicial.rename("contraste").rename_axis("termo").reset_index().assign(sentido="maior em 1890–1895"),
    termos_periodo_final.rename("contraste").rename_axis("termo").reset_index().assign(sentido="maior em 1896–1901"),
], ignore_index=True)
display(tabela_contrastes[["termo", "contraste", "sentido"]].round(3))

O contraste depende do conteúdo repetitivo criado para os dados fictícios e da
distribuição dos temas pelos anos. Para interpretar um termo, retorne aos textos
e concordâncias; o peso não explica por que ele aparece.

## U05-A04 — Atividade integrada — matriz documentada

**Modalidade:** dupla. **Tempo:** 35 minutos.

1. documente tokenização e vocabulário;
2. construa uma matriz documento-termo;
3. compare contagem e TF-IDF em dois documentos;
4. identifique dois termos distintivos entre subconjuntos;
5. leia os trechos e registre uma interpretação e uma perda do modelo.

**Minha regra e matriz:** Escreva aqui.

**Termos e trechos examinados:** Escreva aqui.

**Perdas da representação:** Escreva aqui.

Leve a matriz e suas regras ao Notebook 04. A partir delas calcularemos Jaccard
e cosseno e as compararemos à distância de edição entre versões.